# 🧹 02: Preprocessing Tests
**GUARDIAN-NLP** | Tests and validates the text cleaning pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocess.cleaner import TextCleaner
from src.preprocess.label_encoder import label_to_multihot, LABEL_COLS

sns.set_theme(style='darkgrid')
cleaner = TextCleaner()
print('Modules loaded.')

## 1. TextCleaner Unit Tests

In [ ]:
test_cases = [
    ('<b>Hello</b> world! Visit https://example.com for more.', 'HTML + URL removal'),
    ('@user1 @user2 I hate you all! #hate', 'Mention + hashtag handling'),
    ('I feel so sad 😢💔 and hopeless', 'Emoji conversion'),
    ('ok', 'Short text (should return None)'),
    ('', 'Empty string (should return None)'),
    ('STOP RIGHT NOW!!! You will REGRET this!!!', 'Uppercase normalization'),
    ('The quick brown fox jumps over the lazy dog.', 'Normal sentence'),
]

for text, desc in test_cases:
    result = cleaner.clean(text)
    print(f'[{desc}]')
    print(f'  Input:  {repr(text[:80])}')
    print(f'  Output: {repr(result)}')
    print()

## 2. Label Encoding Tests

In [ ]:
label_tests = [
    'normal',
    'depressive',
    'hate_speech',
    'violent',
    'violent|depressive',
    'hate_speech|violent',
]

print(f'{"Label String":<30} {" ".join(LABEL_COLS)}')
print('-' * 70)
for label in label_tests:
    vec = label_to_multihot(label)
    print(f'{label:<30} {vec}')

## 3. Pipeline Test on Full CSV

In [ ]:
import os

combined = '../data/raw/combined_raw.csv'
if os.path.exists(combined):
    df = pd.read_csv(combined).head(1000)  # test on first 1000 rows
    print(f'Loaded {len(df)} rows')
    df_clean = cleaner.clean_dataframe(df)
    print(f'After cleaning: {len(df_clean)} rows')
    print(f'Null rate: {df_clean["text"].isnull().mean():.1%}')
    
    # Check word counts
    df_clean['word_count'] = df_clean['text'].apply(lambda x: len(str(x).split()))
    print(f'\nWord count stats:')
    print(df_clean['word_count'].describe())
else:
    print('Run data_merger.py first.')

## 4. BERT Tokenizer Test

In [ ]:
from src.preprocess.tokenizer_utils import get_tokenizer, tokenize_batch, decode_tokens

tokenizer = get_tokenizer()
sample_texts = [
    'I can not take it anymore, everything feels pointless.',
    'You deserve to suffer for what you did.',
    'Had a wonderful day with friends at the park!',
]

encodings = tokenize_batch(sample_texts, tokenizer=tokenizer)
print(f'input_ids shape: {encodings["input_ids"].shape}')    # [3, 128]
print(f'attention_mask shape: {encodings["attention_mask"].shape}')

print('\nDecoded tokens (sample 0):')
print(decode_tokens(encodings['input_ids'][0], tokenizer=tokenizer))